# Random forest: daily streamflow in the Pacific Northwest

Trains a `RandomForestRegressor` on the shared `pnw_model_dataset.csv` and scores it against the climatology baseline.

In [ ]:
# 1. Setup
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

SEED = 16

In [ ]:
# 2. Load the data
csv_path = "../datacleaning-pipeline/pnw_model_dataset.csv"
df = pd.read_csv(csv_path, dtype={"gauge_id": str}, parse_dates=["date"])

print(df.shape)
print(df["split"].value_counts())

In [ ]:
# 3. Inputs & target
# Raw columns, not the `_z` ones: a tree splits on thresholds, so scaling changes nothing, and raw units keep the importance chart readable.
input_cols = [
    "prcp", "tmax", "srad", "vp", "dayl",
    "drainage_area_km2", "elevation_m", "slope_m_per_km", "frac_forest",
    "mean_precip_mm_day", "mean_pet_mm_day",
]
target_col = "log_runoff"

In [ ]:
# 4. Split
# Train = Oct 2006 – Sep 2011, test = Oct 2011 – Sep 2014, from the `split` column. Warm-up rows are not used; they exist for the LSTM.
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

X_train, y_train = train[input_cols], train[target_col]
X_test, y_test = test[input_cols], test[target_col]

print(len(X_train), "train rows,", len(X_test), "test rows")

In [ ]:
# 5. Baseline
# Climatology: for each basin and calendar day, the mean runoff over the training years. Keyed on month-day so leap years do not shift the calendar.
train_mmdd = train["date"].dt.strftime("%m-%d")
test_mmdd = test["date"].dt.strftime("%m-%d")

climatology = (
    train.assign(mmdd=train_mmdd)
    .groupby(["gauge_id", "mmdd"])["runoff_mm_day"]
    .mean()
    .rename("baseline_runoff")
)

test = test.assign(mmdd=test_mmdd).join(climatology, on=["gauge_id", "mmdd"])
print(test["baseline_runoff"].isna().sum(), "test rows with no baseline (should be 0)")

In [ ]:
# 6. Train
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

# Settings: 200 trees, unlimited depth, seed 16, all other scikit-learn defaults. 200 trees follows Shanko & Melesse (2026).


In [ ]:
# 7. Predict
# Undo the log with `np.expm1`; clip at zero so all three models treat negative predictions the same way.
pred_log = rf.predict(X_test)
test = test.assign(
    pred_runoff=np.clip(np.expm1(pred_log), a_min=0.0, a_max=None),
    obs_runoff=test["runoff_mm_day"],
)

print(test[["obs_runoff", "pred_runoff", "baseline_runoff"]].describe().round(2))

In [ ]:
# 8. Scores
# NSE and RMSE on the original scale (mm/day), pooled and per-basin median. High-flow day = runoff above the basin's 95th percentile over the training years.
def nse(obs, pred):
    return 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2)

def rmse(obs, pred):
    return np.sqrt(np.mean((obs - pred) ** 2))

cutoff = train.groupby("gauge_id")["runoff_mm_day"].quantile(0.95).rename("cutoff")
test = test.join(cutoff, on="gauge_id")
test["obs_high"] = test["obs_runoff"] > test["cutoff"]

def score(pred_col):
    obs, pred = test["obs_runoff"], test[pred_col]
    per_basin = test.groupby("gauge_id").apply(lambda g: nse(g["obs_runoff"], g[pred_col]))
    pred_high = pred > test["cutoff"]
    return {
        "NSE pooled": nse(obs, pred),
        "NSE per-basin median": per_basin.median(),
        "RMSE mm/day": rmse(obs, pred),
        "high-flow recall": (pred_high & test["obs_high"]).sum() / test["obs_high"].sum(),
        "high-flow precision": (pred_high & test["obs_high"]).sum() / max(pred_high.sum(), 1),
    }

results = pd.DataFrame({
    "random forest": score("pred_runoff"),
    "climatology": score("baseline_runoff"),
}).round(3)
results

In [ ]:
# 9. Importance


In [ ]:
# 10. Plots
